# Structured tool calling-- teaching an LLM to use external functions safely and predictably instead of relying on hallucinated knowledge.

This is one of the highest-value skills in current AI engineering roles because alsmost every production AI system interacts with APIs, databases, search engines, or business services.


# Why tool calling matters?

A language model only knows what is inside:
    * Its training data.
    * The current prompt.
    * Retrieved context(RAG).

It cannot reliably know:
    * Today's weather.
    * Inventory levels.
    * Bank balances.
    * Databse contents.
    * ERP records.
    * Robot sensor values.
    * Production machine status.
Instead of guessing, the model should call a tool.

Example:

```
    User
      |
      V
     LLM
      |
      V
    Should I answer?
    or 
    Should I call a tool?
      |
      V
    Weather API
      |
      V
     LLM
      |
      V
Final Answer
```
Production AI systems are primarily decision systems, not just text generators.


# What is a Tool?
A tool is simply a normal Python function that the LLM can invoke.


```Python
def get_inventory(product_id: str):
    ...
```

or 
```Python
def search_documents(query:str):
    ...
```
or 
```Python
def create_work_order(machine_id:str):
    ...
```

The LLM decides:
* Whether to call it.
* What arguments to provide.
* How to use the returned result.


# Tool calling Architecture:
```
            User
              |
              V
        FastAPI Endpoint
              |
              V
        LangGraph Agent
              |
              V
        Decide Next Action
              |
      ----------------
      |              |
No Tool Needed Tool Required
      |              |
      v              V
LLM Response    Execute Python Tool
                     |
                     v
                Structured Result
                     |
                     V
                Return to LLM
                     |
                     v
                Final Response 
```


              

# Good Tool Design:
A production tool should have:
    * One clear responsibility.
    * Typed inputs.
    * Typed outputs.
    * Validation.
    * Predictable behavior.
    * Error Handling.

Good Tool:
```Python
def get_machine_status(machine_id: str)->dict:
```
Bad Tool:

```Python
def do_everything(prompt):
```
Small, focused tools are easier to test and reuse.

# Why Structured Inputs Matter:
Instead of passing free-form text:

`
Repair machine 15 tomorrow morning.
`

Usw Schema:

```Python
class MaintenanceRequest(BaseModel):
    machine_id: str
    date: str
    priority: str
```

Advantagesd:

* Validation.
* Autocomplete.
* Easier debugging.
* Safer execution.
* Cleaner APIs.

This is why Pydantic is central to modern AI engineering.


# Tool Calling Workflow:
Imagine a predictive maintenance assitant.

User:

`
Machine A12 has high pressure and vibration.
`
# Workflow:

```
    User
     |
     V
    LLM
     |
     v
    Call:
retrieve_machine_history()
     |
     v
History Retrieved
     |
     v
 call:
 retriev_manual()
     |
     v
Return configuration
     |
     v
Final Answer

```

The LLM coordinates the workflow while specialized tools handle the operational tasks.

# Tool Categories:
Retrieval Tools
Examples:
* Vector database search.
* Knowledge base search.
* PDF retrieval.

# API Tools:
Examples:
* Weather API.
* GitHub API.
* Slack API.
* Jira API.

# Database Tools:
Examples:

* PostgreSQL
* SQLite.
* MongoDB.

# Robotics Tools:
Given your mechatronics and robotics background, think about tools like:

```
Read Camera frame
        |
        v
     Read IMU
        |
        v
    Read Force Sensor
        |
        v
    Read PLC status
        |
        v
   Move robot arm
        |
        v
    Emergency stop
```
An agent should never invent a sensor value. It should request the real value from a tool.

# Computation Tools:

Examples:

    * Calculator.
    * Optimization.
    * Simulation.
    * Path Planning.


# Example Agent Flow:

```
        User:
Generate today's maintenance report.
            |
            v
          Agent
            |
            v
    Get today's machine logs
            |
            v
      Summarize failures
            |
            v
       Calculate downtime
            |
            v
     Generate recomendations
            |
            v
      Save Report
            |
            v
        Return report
```
Each step is seprate tool.


# Tool Errors 
Production systems must expect failures.
Possible Issues:

    * API timeout.
    * Database Unavailable.
    * Invalid arguments.
    * Permission denied.
    * Rate limit exceeded.

The LLM should receive structured error infromation instead of a Python traceback.


Example

```JSON
{
    "status":"error",
    "reason":"Machine not found"
}
```
The model can then explain the issue to the user and, where appropriate, suggest corrective action.

# Security Considerations:

Never allow unrestricted tool execution.

BAD:

`
Run any shell command.
`

Better:

* Expose only approved tools.
* Validated arguments.
* Authenticate users.
* Authorize sensitive operations.
* Log tool usage.
* Audit critical actions.

For example, creating a work order should require authenticate even if reading documentation does not.

# Small Coding Exercise:

Create three Pydantic-based tools:

```Python
get_machine_status(machine_id)

search_manual(query)

create_work_order(machine_id,issue)
```
Requirements:

* Validate inputs.
* Return structured dictionaries or Pydantic models.
* Handle invalid input gracefully.
* Write unit test for each tool.


# Tentative Interview Questions:

1. What is tool calling?
Answer:
Tool calling allows an LLM to invoke external functions,APIs, or services to obtain real-world information or perform actions instead of relying solely on its internal knowledge.

2. Why use Pydantic with tool calling?
Answer:

Pydantic validates inputs and outputs, enforces data contracts, improves reliability, reduces runtime errors, and makes tool interfaces easier to maintain and test.

3. What is the difference between RAG and tool calling?

Answer:

* RAG retrieves information for the model to read and reason about.
* Tool calling executes actions or queries external systems, such as databases, APIs, or business services.

Many production systems combine both: retrieve relevent context with RAG, then use tools to perform operations or fetch live data.

4. Why should tools have a single responsibility?
Answer:
Small, focused tools are easier to understand, test, secure, and compromise into larger workflows. They also make it easier for the LLM to select the correct tool.


5. What security measures should be applied to AI tools?

Answer:

 * Validate inputs.
 * Authenticate users.
 * Authorize sensitive actions.
 * Restrict available tools.
 * Log tool usage.
 * Handle errors safely.
 * Avoid exposing arbitrary code execution.
 